# Temporal NLG Fine-Tuning (LoRA) on Colab GPU
- Base model: `microsoft/Phi-4-mini-instruct` (can swap)
- Adapter: QLoRA (bitsandbytes) for single-GPU training
- Task: Instruction-to-text (temporal explanations)
- Outputs: LoRA adapter + merged model (optional)
- Data format: JSONL with {"instruction", "input", "output"}

In [ ]:
# If you're on Colab, set your runtime to GPU (T4/A100 preferred)
import os, platform, torch, json, textwrap, subprocess, sys
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Install deps (pin to newer transformer for Phi-4)
!pip -q install -U \
  "transformers==4.45.2" \
  "datasets==2.20.0" \
  "accelerate==0.34.2" \
  "bitsandbytes==0.44.1" \
  "peft==0.13.0" \
  "trl==0.9.4"

> If you previously imported transformers in this runtime, restart runtime after this install so the newer version (>=4.45) is picked up; then rerun from the top.

### If tokenizer load fails (cache)
If you see `ModelWrapper`/`data did not match any variant` errors, clear the cached Phi-4 tokenizer and re-run the next cell.

In [ ]:
# Clear cached Phi-4 tokenizer files if they got corrupted
import shutil
from pathlib import Path

cache_dir = Path("/root/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct")
if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print("Cleared", cache_dir)
else:
    print("No cached Phi-4 tokenizer to clear; continuing.")

## Configure paths and model

In [ ]:
from pathlib import Path

# Swap base_model for another small model if desired
base_model = "microsoft/Phi-4-mini-instruct"

# Where your training/validation/test data will live (JSONL) on Drive
data_dir = Path("/content/drive/MyDrive/explanability-for-temporal-graphs/temporal_nlg_data")
train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"
test_path = data_dir / "test.jsonl"

# Output directories on Drive
output_dir = Path("/content/drive/MyDrive/explanability-for-temporal-graphs/temporal_nlg_lora")
merged_dir = Path("/content/drive/MyDrive/explanability-for-temporal-graphs/temporal_nlg_merged")

## Prepare data from processed JSONL splits
Assumes your train/val/test JSONL files already live in your Drive at `data_dir` (no downloading or splitting performed here).

In [ ]:
# Assumes train/val/test JSONLs already exist at data_dir (e.g., on Drive)
data_dir = Path("/content/drive/MyDrive/snet_related/temporal_nlg_data")
data_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "train.jsonl"
val_path = data_dir / "val.jsonl"
test_path = data_dir / "test.jsonl"

if not train_path.exists():
    raise FileNotFoundError(f"Processed train.jsonl not found at {train_path}.")

def _serialize_for_io(inp):
    if isinstance(inp, dict):
        temporal_type = inp.get("temporal_type") or inp.get("type")
        fields = inp.get("fields", inp)
        parts = []
        if temporal_type:
            parts.append(f"type: {temporal_type}")
        for k, v in fields.items():
            if k == "temporal_type":
                continue
            parts.append(f"{k}: {v}")
        return "; ".join(parts)
    return str(inp).strip()

def flatten_file(src: Path, dst: Path):
    if not src.exists():
        return False
    with src.open("r", encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            rec = json.loads(line)
            rec["input"] = _serialize_for_io(rec.get("input", ""))
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return True

flat_train = data_dir / "train_flat.jsonl"
flat_val = data_dir / "val_flat.jsonl"
flat_test = data_dir / "test_flat.jsonl"

flatten_file(train_path, flat_train)
flatten_file(val_path, flat_val)
flatten_file(test_path, flat_test)

# Use flattened files for load_dataset
train_path = flat_train
val_path = flat_val
test_path = flat_test

HAS_VAL = val_path.exists() and val_path.stat().st_size > 0
HAS_TEST = test_path.exists() and test_path.stat().st_size > 0

print("Using processed splits (train/val/test) from", data_dir)
print("Val present:", HAS_VAL, "Test present:", HAS_TEST)

## Load data with Datasets

In [ ]:
from datasets import load_dataset

data_files = {"train": str(train_path)}
if HAS_VAL:
    data_files["validation"] = str(val_path)
if HAS_TEST:
    data_files["test"] = str(test_path)

ds = load_dataset("json", data_files=data_files)
ds

## Build instruction-formatting function
Format: `<instruction>\n\n<input>\n\n### Response:\n<output>`
- If `input` is structured (dict with fields/temporal_type), we serialize its fields.

In [ ]:
def serialize_input(inp):
    if isinstance(inp, dict):
        temporal_type = inp.get("temporal_type") or inp.get("type")
        fields = inp.get("fields", inp)
        parts = []
        if temporal_type:
            parts.append(f"type: {temporal_type}")
        for k, v in fields.items():
            if k == "temporal_type":
                continue
            parts.append(f"{k}: {v}")
        return "; ".join(parts)
    return str(inp).strip()

def format_example(example):
    instruction = example.get("instruction", "Generate a temporal explanation").strip()
    inp = serialize_input(example.get("input", ""))
    output = example["output"].strip()
    example["text"] = f"{instruction}\n\n{inp}\n\n### Response:\n{output}"
    return example

if HAS_VAL or HAS_TEST:
    ds = ds.map(format_example)
else:
    ds["train"] = ds["train"].map(format_example)

ds["train"][0]["text"]

## Load tokenizer and model (QLoRA ready)

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

# Use slow tokenizer + trust_remote_code to avoid fast-tokenizer model wrapper errors
# If you hit a cache corruption error, clear `/root/.cache/huggingface/hub/models--microsoft--Phi-4-mini-instruct` and re-run.
tokenizer = AutoTokenizer.from_pretrained(
    base_model,
    trust_remote_code=True,
    use_fast=False,
)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

bnb_config = bnb.nn.Linear8bitLt

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Tokenize dataset

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="longest",
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

if HAS_VAL or HAS_TEST:
    tokenized = ds.map(tokenize, batched=True, remove_columns=ds["train"].column_names)
else:
    tokenized = {}
    tokenized["train"] = ds["train"].map(tokenize, batched=True, remove_columns=ds["train"].column_names)

tokenized

## Training configuration (real run defaults)
- If `VAL_RATIO` > 0, we evaluate every few steps; if `VAL_RATIO` = 0, we skip eval.
- If `TEST_RATIO` > 0, a held-out test split is saved and evaluated after training.
- Uses small batch with gradient accumulation for T4/A100.
- Increase `num_train_epochs` or steps if you need more convergence.
- Adjust `max_length` in tokenization if your sequences are longer.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, __version__ as hf_version
from packaging import version

if version.parse(hf_version) < version.parse("4.45.0"):
    raise RuntimeError(
        f"transformers {hf_version} is too old; after rerunning the install cell, restart runtime to load >=4.45.0."
    )

# Bump per-device batch to use more GPU (adjust down if OOM)
per_device_train_batch_size = 4
per_device_eval_batch_size = 4

max_steps = 120  # cap total steps
num_epochs = 1

automatic_eval = True
if automatic_eval:
    evaluation_strategy = "steps"
    eval_steps = 20  # run val every 20 steps
else:
    evaluation_strategy = "no"
    eval_steps = None

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=num_epochs,
    max_steps=max_steps,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=4,  # effective batch 16 tokens-per-batch with batch_size=4
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.02,
    logging_strategy="steps",
    logging_steps=10,  # log train loss every 10 steps
    logging_first_step=True,
    evaluation_strategy=evaluation_strategy,
    eval_steps=eval_steps,  # val loss every 20 steps
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    fp16=False,
    bf16=True,
    report_to="none",
    disable_tqdm=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized.get("validation") if automatic_eval else None,
    data_collator=data_collator,
)

trainer

## Train (LoRA)

In [ ]:
train_result = trainer.train()
train_result

## Evaluate on test (if available)

In [ ]:
if HAS_TEST:
    test_metrics = trainer.evaluate(eval_dataset=tokenized["test"])
    print("Test metrics:", test_metrics)
else:
    print("No test split configured; skip test eval.")

## Save adapter and (optionally) merge

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved LoRA adapter to", output_dir)

# Optional: merge LoRA into base weights (for export)
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto",
    load_in_4bit=False,
    torch_dtype=torch.bfloat16,
)
merged = PeftModel.from_pretrained(base, output_dir)
merged = merged.merge_and_unload()
merged_dir.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print("Merged model saved to", merged_dir)

In [ ]:
# Zip outputs for download from Colab (no Drive needed)
# Run after training completes.
import shutil
from pathlib import Path

local_lora = Path("/content/temporal_nlg_lora")
local_merged = Path("/content/temporal_nlg_merged")

# Copy current outputs to local folders (they may already be local if you pointed output_dir there)
if output_dir.exists():
    shutil.copytree(output_dir, local_lora, dirs_exist_ok=True)
if merged_dir.exists():
    shutil.copytree(merged_dir, local_merged, dirs_exist_ok=True)

!zip -r /content/temporal_nlg_lora.zip /content/temporal_nlg_lora
!zip -r /content/temporal_nlg_merged.zip /content/temporal_nlg_merged
print("Zips written to /content; use the Colab file browser to download.")

In [ ]:
# Direct download a specific file from Colab
from google.colab import files
files.download('/content/temporal_nlg_merged/model-00001-of-00002.safetensors')

## Quick inference test

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=str(merged_dir),
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

prompt = "Generate a temporal explanation\n\nentity: Apollo 11; event: moon landing; date: 1969-07-20; context: NASA mission\n\n### Response:\n"
gen = pipe(prompt, max_new_tokens=60, do_sample=True, temperature=0.7)[0]["generated_text"]
print(gen)

## (Optional) Mount Google Drive to persist outputs

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/temporal_nlg_lora /content/drive/MyDrive/
# !cp -r /content/temporal_nlg_merged /content/drive/MyDrive/

## Tips to scale up
- Increase `num_train_epochs`, dataset size, and `max_length` if your facts are longer.
- Use gradient_accumulation to fit larger batches.
- Monitor loss and add evaluation metrics (BLEU/ROUGE) by decoding predictions on val set.
- For larger models, ensure A100 GPU and possibly reduce `max_length` or use 4-bit with smaller `r`.

## Milestone 1 handoff (training + inference)
- Colab run finished: `TrainOutput(global_step=50, training_loss=1.78, runtime≈38.7m)`; use these numbers in docs.
- LoRA adapter: copy/download to `models/temporal_nlg_lora` (or zipped from Colab) and optionally merged weights to `models/temporal_nlg_merged`.
- Smoke tests to complete milestone: (1) local inference via new CLI, (2) run `experiments/m1_e2_llm_nlg/run_eval.py` on held-out examples, (3) record metrics in docs/RESULTS_M1.md.
- If no GPU locally, run inference with `--no-4bit` (CPU) and keep `max_new_tokens` small (≤64).

In [ ]:
# Smoke-test inference using the saved adapter (runs on CPU if --no-4bit)
# Adjust paths if your adapter/merged weights are stored elsewhere.
from pathlib import Path
import subprocess, sys

adapter_path = Path("models/temporal_nlg_lora")
base_model = "microsoft/phi-4-mini-instruct"

if not adapter_path.exists():
    print(f"Adapter path not found: {adapter_path}. Place your downloaded LoRA there.")
else:
    cmd = [
        sys.executable,
        "examples/milestone1/lora_inference.py",
        "--instruction",
        "Generate a temporal explanation",
        "--input-text",
        "entity: Apollo 11; event: moon landing; date: 1969-07-20; context: NASA mission",
        "--adapter-path",
        str(adapter_path),
        "--base-model",
        base_model,
        "--no-4bit",  # remove if you have GPU 4-bit available
        "--max-new-tokens",
        "64",
    ]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
    print("stdout:\n", proc.stdout)
    print("stderr:\n", proc.stderr)

## Inference recipes (GPU vs CPU)
Use the standalone CLI so training stays separate. Set `adapter_path` to where you stored the Colab-trained LoRA.

In [ ]:
# GPU inference (4-bit, requires CUDA)
from pathlib import Path
import subprocess, sys

adapter_path = Path("models/temporal_nlg_lora")
base_model = "microsoft/phi-4-mini-instruct"

if not adapter_path.exists():
    print(f"Adapter path not found: {adapter_path}")
else:
    cmd = [
        sys.executable,
        "examples/milestone1/lora_inference.py",
        "--instruction",
        "Generate a temporal explanation",
        "--input-text",
        "entity: Apollo 11; event: moon landing; date: 1969-07-20; context: NASA mission",
        "--adapter-path",
        str(adapter_path),
        "--base-model",
        base_model,
        # GPU path: keep 4-bit enabled (default)
        "--max-new-tokens",
        "64",
    ]
    print("Running (GPU, 4-bit):", " ".join(cmd))
    proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
    print("stdout:\n", proc.stdout)
    print("stderr:\n", proc.stderr)

In [ ]:
# CPU inference (no 4-bit quantization)
from pathlib import Path
import subprocess, sys

adapter_path = Path("models/temporal_nlg_lora")
base_model = "microsoft/phi-4-mini-instruct"

if not adapter_path.exists():
    print(f"Adapter path not found: {adapter_path}")
else:
    cmd = [
        sys.executable,
        "examples/milestone1/lora_inference.py",
        "--instruction",
        "Generate a temporal explanation",
        "--input-text",
        "entity: Apollo 11; event: moon landing; date: 1969-07-20; context: NASA mission",
        "--adapter-path",
        str(adapter_path),
        "--base-model",
        base_model,
        "--no-4bit",
        "--max-new-tokens",
        "64",
    ]
    print("Running (CPU, full precision):", " ".join(cmd))
    proc = subprocess.run(cmd, check=False, capture_output=True, text=True)
    print("stdout:\n", proc.stdout)
    print("stderr:\n", proc.stderr)